# GIC 2026 — QRC on qBraid quantum hardware

**Phase 3 hardware proof for Track 1 (SPY realized-volatility forecasting).**

This notebook runs the *same* 5-qubit gate-based quantum reservoir that wins in simulation as **real quantum jobs on qBraid**, rebuilds the Z/ZZ reservoir features from shot counts, and scores the volatility forecast against the classical baselines.

Run order (cheapest first):
1. Connect + list devices.
2. **Bell-state smoke test** — confirm submission works before spending anything.
3. Dry-run the QRC experiment (no jobs).
4. Full run on the **free statevector simulator** `qbraid:qbraid:sim:qir-sv`.
5. *(Optional, spends credits)* small run on a real QPU.

> **Honest framing:** in simulation QRC ≈ ESN ≈ Persistence. The hardware claim is *the quantum reservoir executes on real QPUs, its features survive device noise (high fidelity vs statevector), and the forecast stays competitive with the best classical baselines* — not that it beats everything. The saved job ids are the evidence of real hardware execution.

In [ ]:
# Run from the repo root (clone this repo into qBraid Lab first).
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import qbraid
from qbraid.runtime import QbraidProvider
print('qbraid', qbraid.__version__)

In [ ]:
# Inside qBraid Lab the API key is already configured. Locally, set it once:
#   export QBRAID_API_KEY=...   (from account.qbraid.com > Account > API Keys)
provider = QbraidProvider()

devices = provider.get_devices()
for dev in devices:
    print(dev)

In [ ]:
# Pick the free statevector simulator and inspect it.
DEVICE_ID = 'qbraid:qbraid:sim:qir-sv'
device = provider.get_device(DEVICE_ID)
device.metadata()

## 1. Bell-state smoke test

Submit one tiny circuit to confirm auth + submission + result parsing work before touching the real pipeline.

In [ ]:
from qiskit import QuantumCircuit

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

job = device.run(bell, shots=1000)
result = job.result()
print('counts:', result.data.get_counts())
print('job id:', getattr(job, 'id', getattr(job, 'job_id', '?')))

## 2. Dry-run the QRC experiment (no jobs submitted)

Builds the reservoir circuits and prints the job/shot footprint so you know exactly what a real run will cost.

In [ ]:
from experiments.qbraid_hardware_qrc import run, _build_argparser

args = _build_argparser().parse_args([
    '--device', DEVICE_ID,
    '--max-test', '120',
    '--shots', '1024',
    '--dry-run',
])
run(args)

## 3. Full run on the free statevector simulator

Trains the readout on exact local features, runs the test window as real qBraid jobs, and reports feature fidelity + the leaderboard. Job ids are saved to `results/qbraid_job_ids.json`.

In [ ]:
args = _build_argparser().parse_args([
    '--device', DEVICE_ID,
    '--max-test', '120',
    '--shots', '1024',
])
run(args)

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('results/qbraid_hardware_summary.csv').round(5))
display(Image('results/qbraid_hardware_qrc.png'))

## 4. (Optional) Real QPU run — spends credits

Keep `--max-test` small. `--allow-qpu` is the required safety latch. Check the device queue and your credit balance first.

**QuEra Aquila (`aws:quera:qpu:aquila`) and Pasqal Fresnel are analog neutral-atom devices — they do NOT run these gate circuits.** Use a gate QPU. Online gate QPUs observed for this account: `openquantum:ionq:qpu:forte-1` (36q trapped-ion, all-to-all — best fit), `openquantum:iqm:qpu:garnet` (20q), `openquantum:rigetti:qpu:cepheus-1-108q` (107q), `openquantum:aqt:qpu:ibex-q1` (12q). Run the `get_devices()` cell above (or `experiments/qbraid_list_devices.py --online --gate`) to confirm what is live.

In [ ]:
# UNCOMMENT to run on a gate QPU (this consumes credits):
# QPU_ID = 'openquantum:ionq:qpu:forte-1'   # trapped-ion, all-to-all — NOT QuEra (analog)
# print(provider.get_device(QPU_ID).metadata())   # check status / queue first
# args = _build_argparser().parse_args([
#     '--device', QPU_ID,
#     '--max-test', '40',
#     '--shots', '1024',
#     '--max-batch', '20',
#     '--allow-qpu',
# ])
# run(args)